# AgriSwin-MoE: train on Colab

Runs the **same** `train.py`, recipe and frozen splits as the laptop, on a Colab
L4 or A100. Nothing about training lives in this notebook: it prepares the
runtime and starts the queue runner, so a Colab result can share a table with a
laptop result.

**Before you run anything**

1. **Runtime > Change runtime type > L4 GPU** (or A100 GPU) > Save.
2. *Recommended:* add a `GITHUB_TOKEN` secret (the key icon in the left sidebar)
   so each result is pushed to GitHub the moment its run finishes. Without it,
   results are kept on your Google Drive. How to make the token:
   `docs/NEXT_STEPS.md`, Part C.
3. **Runtime > Run all**, and keep this tab open while it trains.

**If the runtime disconnects**, just *Run all* again: finished runs are skipped
and an interrupted one resumes from its last finished epoch, because
checkpoints live on your Drive.

In [1]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
NVIDIA L4, 23034 MiB, 580.82.07


In [2]:
# 2. Connect Google Drive. Checkpoints and the image cache live there, so they
#    survive a disconnect. The first time, Colab asks you to allow access.
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# 3. Get the latest code
import os
REPO = "/content/agriswin-moe"
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/amritaj312-max/agriswin-moe.git {REPO}
%cd {REPO}
!git log --oneline -1

Cloning into '/content/agriswin-moe'...
remote: Enumerating objects: 294, done.
remote: Counting objects: 100% (294/294), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 294 (delta 144), reused 279 (delta 129), pack-reused 0 (from 0)
Receiving objects: 100% (294/294), 4.82 MiB | 7.93 MiB/s, done.
Resolving deltas: 100% (144/144), done.
/content/agriswin-moe
8be95db (HEAD -> main, origin/main, origin/HEAD) Re-evaluate the baselines on the frozen 2,551-image PlantDoc set


In [4]:
# 4. Prepare this runtime: exact packages, images verified byte for byte
#    against the laptop's, checkpoints on Drive. The first time this downloads
#    the images (a few minutes) and caches them on Drive; after that it copies
#    the cache (about a minute).
import importlib
import sys
sys.path.insert(0, f"{REPO}/notebooks")
import colab_runtime
importlib.reload(colab_runtime)
runtime = colab_runtime.setup()

GPU NVIDIA L4 (23034 MiB), 12 CPUs, 53 GB RAM -> 8 data-loading workers

No image cache on Drive yet: downloading (a few minutes, once)...

Checking every image against the reference the results were made on...

Writing the image cache to Drive for next time (/content/drive/MyDrive/agriswin/data/agriswin_images.tar) ...
The GITHUB_TOKEN secret cannot push (remote: Permission to amritaj312-max/agriswin-moe.git denied to amritaj312-max.
fatal: unable to access 'https://github.com/amritaj312-max/agriswin-moe.git/': The requested URL returned error: 403); results will be kept on Drive instead. Check the token's repository and its 'Contents: Read and write' permission.

  code         8be95db Re-evaluate the baselines on the frozen 2,551-image PlantDoc set
  packages     timm==1.0.30
  images       verified, from a fresh download (pinned upstream commits)
  checkpoints  /content/drive/MyDrive/agriswin/checkpoints
  results      kept on Drive (no GITHUB_TOKEN secret, so not pushed)


## What to train

A queue file holds one `train.py` argument line per run. `colab_first.txt` is
the first job: EfficientNet-B0 seed 42 again, which must land inside the
laptop's spread (proving the GPU does not move results), then Swin-V2-T seeds 1
and 2, which give Phase 3's ablation control a seed floor of its own.

**Do not change the batch size or learning rate** for the bigger GPU: every row
of a table must share one recipe, and `train.py` refuses a paper run that
changes it. The speed-up comes from the GPU alone.

In [5]:
# 5. Choose the queue
QUEUE = "ml_pipeline/configs/queues/colab_first.txt"  #@param {type:"string"}
!cat {QUEUE}

# The first Colab queue. One train.py argument line per run; finished runs are
# skipped and an interrupted one resumes, so after a disconnect run it again.
#   notebooks/train_on_colab.ipynb runs this file.
#
# 1. EfficientNet-B0, seed 42: the same run as results/efficientnet_b0.json,
#    on a different GPU. It must land inside the laptop's three-seed spread
#    (test accuracy 99.75-99.79%). If it does, the GPU does not move the
#    results and Colab runs may share a table with laptop runs. If it does not,
#    stop and look before training anything else. The shortest run, so it also
#    proves the whole Colab setup end to end before the long ones.
# 2-3. Swin-V2-T, seeds 1 and 2. Phase 3's ablations are compared against
#    plain Swin-V2-T, and a difference only counts if it clears that model's
#    own seed floor -- which needs more than the one seed trained so far.
--config ml_pipeline/configs/baselines/efficientnet_b0.yaml --set run_name=efficientnet_b0_colab
--config ml_pipe

In [ ]:
# 6. Train. Each result is committed, and pushed with the token, as soon as
#    its run finishes. The img/s figure in the progress lines is this GPU's
#    real speed on this recipe.
NUM_WORKERS = runtime["num_workers"]
TRAILER = f"Trained on Colab ({runtime['gpu']})"
!python ml_pipeline/src/training/run_queue.py {QUEUE} --set-all num_workers={NUM_WORKERS} --commit-trailer "{TRAILER}"

[queue 2026-11-02 06:54:29] ml_pipeline/configs/queues/colab_first.txt: 3 run(s), pid 3875
[queue 2026-11-02 06:54:29] efficientnet_b0_colab: starting
run     efficientnet_b0_colab  (fresh start)
model   efficientnet_b0  4.06 M params  weights timm/efficientnet_b0.ra_in1k
data    train 37,992 / val 8,158 / test 8,155
input   256px  mean (0.485, 0.456, 0.406)  std (0.229, 0.224, 0.225)
recipe  AdamW lr 0.0001 wd 0.05  warmup 2 ep, cosine to 1e-06  40 ep x 1187 steps  batch 32  float16
  epoch 1 [ 100/1187]  loss 3.6376  lr 4.21e-06  70 img/s  eta 8m14s
  epoch 1 [ 200/1187]  loss 3.6007  lr 8.42e-06  405 img/s  eta 1m18s
  epoch 1 [ 300/1187]  loss 3.5377  lr 1.26e-05  418 img/s  eta 1m07s
  epoch 1 [ 400/1187]  loss 3.4217  lr 1.68e-05  410 img/s  eta 1m01s
  epoch 1 [ 500/1187]  loss 3.2595  lr 2.11e-05  415 img/s  eta 0m52s
  epoch 1 [ 600/1187]  loss 3.0796  lr 2.53e-05  408 img/s  eta 0m46s
  epoch 1 [ 700/1187]  loss 2.8891  lr 2.95e-05  410 img/s  eta 0m38s
  epoch 1 [ 800/1187] 

In [ ]:
# 7. Every result so far, from every machine. The Splits and Recipe lines at
#    the bottom must say all runs match.
!python ml_pipeline/src/evaluation/summarize_results.py

In [ ]:
!cd /content/agriswin-moe && git pull --rebase && git push